# 01 - CelebA-Spoof Data Preparation

This notebook builds a clean liveness training dataset for Kaggle training:

1. Load annotation table from CSV/JSON/Parquet.
2. Normalize labels to `0=spoof`, `1=live`.
3. Crop faces using dataset bounding boxes (with margin).
4. Resize crops to `80x80` for MiniFASNet-style inputs.
5. Create train/val/test manifests for downstream training.

Paper writing is intentionally delayed until model results are stable.


## Runtime Setup (Kaggle)

If dependencies are missing in Kaggle, run the cell below once.


In [ ]:
# !pip install -q -r /kaggle/working/Face_Anti_Spoofing_Biometric/requirements-kaggle.txt

In [ ]:
from pathlib import Path
import sys

# Update this path if your repository folder name is different in /kaggle/working.
PROJECT_ROOT = Path('/kaggle/working/Face_Anti_Spoofing_Biometric')
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

print('Project root:', PROJECT_ROOT)
print('Exists:', PROJECT_ROOT.exists())

In [ ]:
import pandas as pd

from fas.celeba_spoof_prep import (
    PrepConfig,
    build_manifest_dataframe,
    create_split_manifests,
    crop_faces_from_manifest,
    load_annotation_table,
    save_manifest,
)


## Configure Dataset Paths

Set these values to match your Kaggle dataset structure.

Common pattern:
- `DATASET_ROOT`: root containing image files
- `ANNOTATION_PATH`: annotation table with image path and live/spoof label
- Optional split column and bbox columns


In [ ]:
DATASET_ROOT = Path('/kaggle/input/celeba-spoofing')
ANNOTATION_PATH = DATASET_ROOT / 'labels.csv'  # change as needed

# Required column names in annotation file
IMAGE_REL_COL = 'image_path'  # relative to DATASET_ROOT
LABEL_COL = 'label'           # values that can be mapped to live/spoof

# Optional: if dataset already has explicit split labels like train/val/test
SPLIT_COL = 'split'           # set to None if unavailable

# Optional: bounding box columns (x, y, width, height). Set to None if unavailable.
BBOX_COLS_XYWH = ('bbox_x', 'bbox_y', 'bbox_w', 'bbox_h')

# Output location for cropped images and manifests
OUTPUT_ROOT = Path('/kaggle/working/celeba_spoof_prepared')
CROP_DIR = OUTPUT_ROOT / 'crops_80x80'
MANIFEST_DIR = OUTPUT_ROOT / 'manifests'

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CROP_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

print('Dataset root:', DATASET_ROOT)
print('Annotation path:', ANNOTATION_PATH)
print('Output root:', OUTPUT_ROOT)

In [ ]:
table = load_annotation_table(str(ANNOTATION_PATH))
print('Rows:', len(table))
print('Columns:', list(table.columns))
display(table.head(3))

## Build Normalized Manifest Frame

`live_values` and `spoof_values` control how labels are mapped.
Adjust these tuples if your dataset uses different values.


In [ ]:
prep_config = PrepConfig(
    dataset_root=str(DATASET_ROOT),
    image_rel_col=IMAGE_REL_COL,
    label_col=LABEL_COL,
    split_col=SPLIT_COL if SPLIT_COL in table.columns else None,
    bbox_cols_xywh=BBOX_COLS_XYWH if all(col in table.columns for col in BBOX_COLS_XYWH) else None,
    live_values=(1, '1', 'live', 'real', True),
    spoof_values=(0, '0', 'spoof', 'fake', False),
    image_size=80,
    bbox_margin_ratio=0.15,
)

print(prep_config)

In [ ]:
if prep_config.split_col:
    split_values = sorted(table[prep_config.split_col].dropna().unique().tolist())
    print('Detected split values:', split_values)
else:
    print('No split column detected. Random split will be created later.')

## Crop Face Images and Save Per-Split Manifests

In [ ]:
saved_paths = {}

if prep_config.split_col:
    # You can customize these mappings if your split labels differ.
    split_mapping = {
        'train': 'train',
        'val': 'val',
        'test': 'test',
    }

    all_frames = []
    for out_name, split_value in split_mapping.items():
        split_df = build_manifest_dataframe(table, prep_config, split_value=split_value)
        print(f'{out_name}: normalized rows =', len(split_df))

        split_crop_dir = CROP_DIR / out_name
        split_crop_dir.mkdir(parents=True, exist_ok=True)

        cropped_df = crop_faces_from_manifest(split_df, prep_config, str(split_crop_dir))
        print(f'{out_name}: cropped rows =', len(cropped_df))

        manifest_path = MANIFEST_DIR / f'{out_name}.csv'
        save_manifest(cropped_df, str(manifest_path))
        saved_paths[out_name] = manifest_path
        all_frames.append(cropped_df)

    merged = pd.concat(all_frames, ignore_index=True)
else:
    normalized_df = build_manifest_dataframe(table, prep_config, split_value=None)
    print('normalized rows =', len(normalized_df))

    cropped_df = crop_faces_from_manifest(normalized_df, prep_config, str(CROP_DIR))
    print('cropped rows =', len(cropped_df))

    split_frames = create_split_manifests(cropped_df, train_ratio=0.8, val_ratio=0.1, seed=42)

    for split_name, split_df in split_frames.items():
        manifest_path = MANIFEST_DIR / f'{split_name}.csv'
        save_manifest(split_df, str(manifest_path))
        saved_paths[split_name] = manifest_path

    merged = cropped_df

print('Saved manifests:')
for name, path in saved_paths.items():
    print(' ', name, '->', path)

In [ ]:
label_counts = merged['label'].value_counts().sort_index()
print('Label distribution (0=spoof, 1=live):')
print(label_counts)

display(merged.head(3))

In [ ]:
# Optional: save one combined manifest for quick debugging
combined_manifest = MANIFEST_DIR / 'all_cropped.csv'
save_manifest(merged, str(combined_manifest))
print('Combined manifest:', combined_manifest)